# Exercise 2: Code Generation with ReAct Prompting

**Tool:** Google Gemini API (`gemini-3.8-flash`) in Google Colab

**ReAct:** the model writes a Thought (reasoning), an Action (code) and an Observation (a check). The code is then run against real test cases, and any failure is fed back as a new Observation.

In [ ]:
%pip install -q google-genai

import os
import re
import json
import textwrap
import traceback
import random
from google import genai
from google.genai import types

# Load the API key (Colab secret named GEMINI_API_KEY, or an environment variable locally)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = genai.Client()
MODEL = "gemini-3.8-flash"

# Helper: send one prompt to the model and return the reply text
def ask(prompt, system=None):
    config = types.GenerateContentConfig(system_instruction=system)
    return client.models.generate_content(model=MODEL, contents=prompt, config=config).text


In [ ]:
# The coding task
TASK = """Write a Python function `summarize_expenses(rows)`.

`rows` is a list of strings, each like "2025-01-01,food,12.50" (date,category,amount).

Requirements:
- Skip blank lines and a header row (a row whose third field is exactly "amount", case-insensitive) silently.
- Any other row that does not have exactly 3 comma-separated fields, or whose amount is not a number,
  must be added (as the original string) to a list of skipped rows.
- Strip whitespace around fields.
- Return a dict with keys:
    "total": float, sum of all valid amounts
    "by_category": dict of category -> float total
    "top_category": the category with the largest total, or None if there are no valid rows
    "skipped": list of the original skipped row strings, in input order
- It must not raise an exception on bad input rows.
- Use only the Python standard library (no third-party imports).
- Formatting: follow PEP 8, include type hints and a short docstring.
"""

# ReAct system prompt: Thought, Action, Observation, then the final code
REACT_SYSTEM = """You are a careful Python engineer who solves coding tasks with the ReAct pattern.
Always answer in exactly this format:

Thought: <reason step by step about what is needed, the edge cases, and the plan>
Action: <a ```python code block with your first implementation>
Observation: <trace 2 or 3 concrete examples through your code by hand, including an edge case,
              and say honestly whether you found any bug>
Thought: <decide whether the code needs changes>
Final Answer: <a single ```python code block with the final code, nothing else after it>"""

In [ ]:
# Test data used to check the generated code (written independently of the model)
test_rows = ["date,category,amount", "2025-01-01,food,12.50", "2025-01-02,rent,1000",
             "2025-01-03, food ,7.5", "", "bad row", "2025-01-04,fun,abc", "2025-01-05,fun,30"]
expected_by_category = {"food": 20.0, "rent": 1000.0, "fun": 30.0}
expected_skipped = ["bad row", "2025-01-04,fun,abc"]

In [ ]:
# ReAct loop: reason, act, then feed the real test results back as the Observation
history = [types.Content(role="user", parts=[types.Part(text=TASK)])]
final_code = None

for round_no in range(1, 4):
    # Thought, Action and mental Observation from the model
    reply = client.models.generate_content(model=MODEL, contents=history,
                                           config=types.GenerateContentConfig(system_instruction=REACT_SYSTEM)).text
    print(f"===== Round {round_no}: model reply =====")
    print(reply)

    # Extract the last python code block (the Final Answer)
    blocks = re.findall(r"```python\s*(.*?)```", reply, flags=re.DOTALL)
    code_str = blocks[-1].strip() if blocks else ""

    # Run the generated code against the test cases
    try:
        namespace = {}
        exec(code_str, namespace)
        result = namespace["summarize_expenses"](test_rows)
        assert abs(result["total"] - 1050.0) < 1e-9, f"total wrong: {result['total']}"
        assert result["by_category"] == expected_by_category, f"by_category wrong: {result['by_category']}"
        assert result["top_category"] == "rent", f"top_category wrong: {result['top_category']}"
        assert result["skipped"] == expected_skipped, f"skipped wrong: {result['skipped']}"

        empty = namespace["summarize_expenses"]([])
        assert empty == {"total": 0, "by_category": {}, "top_category": None, "skipped": []}, f"empty input wrong: {empty}"

        only_bad = namespace["summarize_expenses"](["x,y", "2025-01-01,food,NaNish"])
        assert only_bad["skipped"] == ["x,y", "2025-01-01,food,NaNish"], f"only_bad wrong: {only_bad}"
        observation = "All tests passed"
        passed = True
    except Exception:
        observation = traceback.format_exc()
        passed = False

    print(f"\n===== Round {round_no}: real observation =====")
    print(observation)

    # Stop if the tests passed, otherwise send the observation back to the model
    if passed:
        final_code = code_str
        break
    history.append(types.Content(role="model", parts=[types.Part(text=reply)]))
    feedback = (f"Observation from running your code:\n{observation}\n\n"
                "Continue the ReAct loop: write a new Thought explaining what went wrong, "
                "then give a corrected Action and Final Answer in the same format.")
    history.append(types.Content(role="user", parts=[types.Part(text=feedback)]))

In [ ]:
# Show the final generated code
print(final_code)

In [ ]:
# Demonstrate the generated function on new data
exec(final_code)

demo_rows = [
    "date,category,amount",
    "2025-03-01,groceries,54.20",
    "2025-03-02,transport,12",
    "2025-03-03,groceries,31.80",
    "oops",
    "2025-03-04,fun,not_a_number",
    "",
    "2025-03-05,fun,45.00",
]
print("Expense Summary:")
print(json.dumps(summarize_expenses(demo_rows), indent=2))

**Reflection:** asking for a Thought before any code made the model list edge cases first. The model's own Observation is only a mental trace, so feeding back real test results let it fix bugs it could not see by reasoning alone.